# 03 US Model Training & Comparison

This redesigned notebook trains models using a configurable feature set.

Default setting:

```text
PROFILE = transfer
FEATURE_SET = tree
```

## Setup

In [1]:
import sys
sys.path.append('../')

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.model import set_seed, compare_models, MODELS

set_seed(42)

PROFILE = 'transfer'
FEATURE_SET = 'tree'

DATA_PATH = f'../data/processed/{PROFILE}/us_modeling_ready_{FEATURE_SET}.csv'
RESULT_DIR = f'../outputs/results/{PROFILE}'
FIGURE_DIR = f'../outputs/figures/{PROFILE}'

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

## Load Prepared Data

In [2]:
df = pd.read_csv(DATA_PATH)
print('Profile:', PROFILE)
print('Feature set:', FEATURE_SET)
print('Shape:', df.shape)
print('Label distribution:')
print(df['potential_label'].value_counts().sort_index())

X = df.drop(columns=['potential_label']).values
y = df['potential_label'].values
feature_names = df.drop(columns=['potential_label']).columns.tolist()
print(f'\nFeatures: {len(feature_names)}')


Profile: transfer
Feature set: tree
Shape: (72537, 103)
Label distribution:
potential_label
0    24279
1    24174
2    24084
Name: count, dtype: int64

Features: 102


## Cross-Validation Model Comparison

In [ ]:
results_df = compare_models(X, y, cv=5, scale_lr=True)
print(results_df.to_string(index=False))
results_path = f'{RESULT_DIR}/model_comparison_{FEATURE_SET}.csv'
results_df.to_csv(results_path, index=False)
print(f'\nSaved to {results_path}')

C:\Users\User\Desktop\SolarPotentialMapping\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\User\Desktop\SolarPotentialMapping\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\User\Desktop\SolarPotentialMapping\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\User\Desktop\SolarPotentialMapping\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect

## Visualize Model Comparison

In [ ]:
metrics_to_plot = [m for m in ['accuracy', 'f1_macro', 'roc_auc_ovr', 'recall_class2', 'recall_macro']
                   if m in results_df.columns]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4 * len(metrics_to_plot), 4))

for ax, metric in zip(axes, metrics_to_plot):
    sns.barplot(data=results_df, x='model', y=metric, ax=ax, palette='muted')
    ax.set_ylim(0, 1)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=7)

plt.tight_layout()
fig_path = f'{FIGURE_DIR}/03_model_comparison_{FEATURE_SET}.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {fig_path}')

## Business Alignment & Hyperparameter Tuning (Issues #5 & #6)

- **Issue #6**：廣告投放場景 → **Class 2（高潛力）Recall** 是主要業務 KPI，高於 f1_macro
- **Issue #5**：以 `recall_class2` 為 scoring 對最佳業務模型做 **RandomizedSearchCV**（n_iter=30, 5-fold）
- 產出 `best_us_model_{FEATURE_SET}_tuned.pkl` 供後續 SHAP 分析使用

In [ ]:
# ── Issue #6: Business KPI — Class 2 (High Potential) Recall ─────────────────
print('=== Business KPI: recall_class2 (Class 2 = High Potential) ===')
if 'recall_class2' in results_df.columns:
    biz_df = results_df[['model', 'recall_class2', 'f1_macro', 'recall_macro']].sort_values(
        'recall_class2', ascending=False
    )
    print(biz_df.to_string(index=False))
    best_biz_model = biz_df.iloc[0]['model']
    best_f1_model  = results_df.loc[results_df['f1_macro'].idxmax(), 'model']
    print(f'\n[Business pick — max recall_class2] : {best_biz_model}')
    print(f'[Statistical pick — max f1_macro]   : {best_f1_model}')
else:
    print('recall_class2 not found — re-run compare_models after updating src/model.py')
    best_biz_model = results_df.loc[results_df['f1_macro'].idxmax(), 'model']

# ── Issue #5: Hyperparameter Tuning via RandomizedSearchCV ───────────────────
from src.model import tune_model, PARAM_GRIDS

tune_target = best_biz_model if best_biz_model in PARAM_GRIDS else \
              next((m for m in [best_f1_model, 'RandomForest'] if m in PARAM_GRIDS), None)

if tune_target:
    print(f'\n=== Tuning {tune_target}: n_iter=30, 5-fold CV, scoring=recall_class2 ===')
    tuned_estimator, tuned_params, tuned_score = tune_model(
        X, y, model_name=tune_target, n_iter=30, cv=5, scoring='recall_class2'
    )
    tuned_bundle = {
        'model': tuned_estimator,
        'model_name': f'{tune_target}_tuned',
        'feature_names': feature_names,
        'scaler': None,
        'profile': PROFILE,
        'feature_set': FEATURE_SET,
        'tuned_params': tuned_params,
        'tuned_score_recall_class2': float(tuned_score),
    }
    tuned_path = f'{RESULT_DIR}/best_us_model_{FEATURE_SET}_tuned.pkl'
    with open(tuned_path, 'wb') as f:
        pickle.dump(tuned_bundle, f)
    print(f'Saved tuned model -> {tuned_path}')
else:
    print('No model in PARAM_GRIDS — skipping tuning')

## Train Best Model on Full Data & Feature Importance

In [ ]:
# Select model by recall_class2 (business KPI) if available, else fall back to f1_macro
if 'recall_class2' in results_df.columns:
    best_model_name = results_df.loc[results_df['recall_class2'].idxmax(), 'model']
    print(f'Best model by recall_class2 (business KPI): {best_model_name}')
else:
    best_model_name = results_df.loc[results_df['f1_macro'].idxmax(), 'model']
    print(f'Best model by f1_macro: {best_model_name}')

best_model = MODELS[best_model_name]
scaler = None

if best_model_name == 'LogisticRegression':
    scaler = StandardScaler()
    X_train_full = scaler.fit_transform(X)
    best_model.fit(X_train_full, y)
else:
    best_model.fit(X, y)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_).mean(axis=0)
else:
    importances = None

if importances is not None:
    feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances})
    feat_imp = feat_imp.sort_values('importance', ascending=False)
    print(feat_imp.head(20))

    fig, ax = plt.subplots(figsize=(9, 7))
    top = feat_imp.head(20).sort_values('importance')
    ax.barh(top['feature'], top['importance'], color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importance ({best_model_name}, {FEATURE_SET})')
    plt.tight_layout()
    fig_path = f'{FIGURE_DIR}/03_feature_importance_{FEATURE_SET}.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()

    imp_path = f'{RESULT_DIR}/feature_importance_{FEATURE_SET}.csv'
    feat_imp.to_csv(imp_path, index=False)
    print(f'Saved to {imp_path}')

## Hold-out Validation: Confusion Matrix & Report

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

val_model = type(best_model)(**best_model.get_params())
if best_model_name == 'LogisticRegression':
    scaler_val = StandardScaler()
    X_train_s = scaler_val.fit_transform(X_train)
    X_test_s = scaler_val.transform(X_test)
    val_model.fit(X_train_s, y_train)
    y_pred = val_model.predict(X_test_s)
else:
    val_model.fit(X_train, y_train)
    y_pred = val_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Saturated (0)', 'Medium (1)', 'High (2)']))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Saturated', 'Medium', 'High'])
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title(f'Confusion Matrix ({best_model_name}, {FEATURE_SET})')
plt.tight_layout()
fig_path = f'{FIGURE_DIR}/03_confusion_matrix_{FEATURE_SET}.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {fig_path}')

## Save Best Model

In [ ]:
model_bundle = {
    'model': best_model,
    'model_name': best_model_name,
    'feature_names': feature_names,
    'scaler': scaler,
    'profile': PROFILE,
    'feature_set': FEATURE_SET,
}

model_path = f'{RESULT_DIR}/best_us_model_{FEATURE_SET}.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_bundle, f)
print(f'Saved to {model_path}')